# Task 1: build a delta hedge by hand

Task 1: hand-built delta hedge of a short ATM call, with and without fees. Brief: README.md in this folder.

## My predictions
Written [before / after] the first full run.

### A. Why the mean P&L is about zero without fees
Zero drift and zero interest rate, so the premium equals the average payoff and the share gains average zero. Mean P&L is about 0.

### B. Slope of log(std) against log(n)
-1/2. Each interval's error scales with the step size, the errors are roughly independent, so the variance scales like 1/n and the std like 1/sqrt(n).

### C. What fees do as n grows
Each trade costs about sqrt(step size) and there are n trades, so total fees grow like sqrt(n) and the mean P&L falls. The std should stop falling once the fee randomness catches up with the hedging error.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm

S0 = 100.0
STRIKE = 100.0
SIGMA = 0.20
N_DAYS = 21
TRADING_DAYS_PER_YEAR = 252
N_PATHS = 10_000
FEE_RATE = 0.001
FREQUENCIES = [5, 21, 84, 336]  # rebalances over the month
SEED = 0

# Added for this notebook.
FINE_STEPS = 1680  # one fine grid for all frequencies; 1680 is divisible by 5, 21, 84 and 336
GITHUB_USER = "AdamYacin30"  # used in the PNG file names
# Save plots next to this notebook, whether it runs from the repo root or from this folder.
OUT_DIR = Path.cwd() if Path.cwd().name == "task01" else Path("notebooks") / "task01"

## 1. Simulate GBM paths

In [2]:
def simulate_gbm_paths(s0, sigma, n_days, n_steps, n_paths, seed):
    """Simulate zero-drift GBM price paths.

    Args:
        s0: starting price.
        sigma: annualized volatility, as a decimal (0.20, not 20).
        n_days: length of the horizon in trading days.
        n_steps: number of equal time steps across the horizon.
        n_paths: number of paths.
        seed: seed for the random generator. Use np.random.default_rng(seed), not the global state.

    Returns:
        Array of shape (n_paths, n_steps + 1). Column 0 is s0 for every path.
    """
    rng = np.random.default_rng(seed)
    dt = n_days / TRADING_DAYS_PER_YEAR / n_steps  # length of one step, in years
    z = rng.standard_normal((n_paths, n_steps))  # one standard normal shock per path per step
    # Zero drift means dS = sigma * S * dW. Its exact log step is -0.5*sigma^2*dt + sigma*sqrt(dt)*z.
    log_steps = -0.5 * sigma**2 * dt + sigma * np.sqrt(dt) * z
    paths = np.empty((n_paths, n_steps + 1))
    paths[:, 0] = s0
    paths[:, 1:] = s0 * np.exp(np.cumsum(log_steps, axis=1))
    return paths


def coarsen_paths(fine_paths, n_rebalances):
    """Keep every k-th column of fine_paths, so the result has n_rebalances steps.

    Every frequency then uses the same underlying paths, just observed less often.

    Args:
        fine_paths: array of shape (n_paths, fine_steps + 1).
        n_rebalances: number of steps wanted. Must divide fine_steps evenly.

    Returns:
        Array of shape (n_paths, n_rebalances + 1).
    """
    fine_steps = fine_paths.shape[1] - 1
    assert fine_steps % n_rebalances == 0, "the fine grid must divide evenly"
    stride = fine_steps // n_rebalances
    return fine_paths[:, ::stride]

In [3]:
# One simulation on the fine grid. Every frequency below is sliced from these same paths.
fine_paths = simulate_gbm_paths(S0, SIGMA, N_DAYS, FINE_STEPS, N_PATHS, SEED)
print(fine_paths.shape, fine_paths.dtype)

# Quick checks on the simulator.
assert np.all(fine_paths[:, 0] == S0)
# Same seed gives identical paths; a different seed gives different ones.
a = simulate_gbm_paths(S0, SIGMA, N_DAYS, 5, 10, seed=1)
assert np.array_equal(a, simulate_gbm_paths(S0, SIGMA, N_DAYS, 5, 10, seed=1))
assert not np.array_equal(a, simulate_gbm_paths(S0, SIGMA, N_DAYS, 5, 10, seed=2))
# Slicing keeps the start and end prices.
for n in FREQUENCIES:
    coarse = coarsen_paths(fine_paths, n)
    assert coarse.shape == (N_PATHS, n + 1)
    assert np.array_equal(coarse[:, -1], fine_paths[:, -1])
# Zero drift: the mean final price is S0 within 4 standard errors.
s_final = fine_paths[:, -1]
assert abs(s_final.mean() - S0) < 4 * s_final.std(ddof=1) / np.sqrt(N_PATHS)

(10000, 1681) float64


## 2. Black-Scholes price and delta (zero interest rate)

In [4]:
def bs_call_price(spot, strike, sigma, tau):
    """Black-Scholes price of a European call with zero interest rate.

    Args:
        spot: underlying price. A float or an array.
        strike: strike price.
        sigma: annualized volatility.
        tau: time to expiry in years, zero or more. Broadcasts with spot.
            At tau == 0 the price is the intrinsic value max(spot - strike, 0).

    Returns:
        Call price, with the same shape as spot.
    """
    spot = np.asarray(spot, dtype=float)
    tau = np.asarray(tau, dtype=float)
    # Where tau is 0, use 1 instead, only so the formula never divides by zero.
    # Those entries are replaced by the intrinsic value on the last line.
    safe_tau = np.where(tau > 0, tau, 1.0)
    vol_sqrt_t = sigma * np.sqrt(safe_tau)
    d1 = (np.log(spot / strike) + 0.5 * sigma**2 * safe_tau) / vol_sqrt_t
    d2 = d1 - vol_sqrt_t
    price = spot * norm.cdf(d1) - strike * norm.cdf(d2)
    intrinsic = np.maximum(spot - strike, 0.0)
    return np.where(tau > 0, price, intrinsic)


def bs_call_delta(spot, strike, sigma, tau):
    """Black-Scholes delta of a European call with zero interest rate.

    Same arguments as bs_call_price. Returns delta, with the same shape as spot.
    At tau == 0, delta is 1 if spot > strike and 0 otherwise.
    """
    spot = np.asarray(spot, dtype=float)
    tau = np.asarray(tau, dtype=float)
    safe_tau = np.where(tau > 0, tau, 1.0)  # same trick as in bs_call_price
    vol_sqrt_t = sigma * np.sqrt(safe_tau)
    d1 = (np.log(spot / strike) + 0.5 * sigma**2 * safe_tau) / vol_sqrt_t
    delta = norm.cdf(d1)
    delta_at_expiry = np.where(spot > strike, 1.0, 0.0)
    return np.where(tau > 0, delta, delta_at_expiry)

In [5]:
# Sanity anchors: about 2.30 and about 0.51.
tau0 = N_DAYS / TRADING_DAYS_PER_YEAR
price0 = float(bs_call_price(S0, STRIKE, SIGMA, tau0))
delta0 = float(bs_call_delta(S0, STRIKE, SIGMA, tau0))
print(f"price {price0:.4f}, delta {delta0:.4f}")
assert abs(price0 - 2.30) < 0.01
assert abs(delta0 - 0.51) < 0.01

# At expiry (tau = 0): intrinsic value and a 0-or-1 delta, with no NaNs.
assert float(bs_call_price(110.0, STRIKE, SIGMA, 0.0)) == 10.0
assert float(bs_call_price(90.0, STRIKE, SIGMA, 0.0)) == 0.0
assert float(bs_call_delta(110.0, STRIKE, SIGMA, 0.0)) == 1.0
assert float(bs_call_delta(90.0, STRIKE, SIGMA, 0.0)) == 0.0

price 2.3030, delta 0.5115


## 3. P&L of the hedged short call

Write the single-path version with a plain for-loop first, then the vectorized version, then check that they agree.

In [6]:
def hedge_pnl_single_path(path, strike, sigma, n_days, fee_rate,
                          hedge=True, charge_initial=True, charge_unwind=True):
    """Final P&L of one hedged short call, computed with a plain for-loop.

    The seller sells one call at its Black-Scholes price at time 0 and holds delta shares,
    rebalancing at every time point in the path.

    Args:
        path: array of shape (n_steps + 1,), prices at the rebalancing times.
        strike: strike price.
        sigma: volatility used for pricing and hedging.
        n_days: horizon in trading days.
        fee_rate: proportional fee on the value of every trade.
        hedge: if False, never hold any shares (the unhedged short call).
        charge_initial: if True, the first purchase of shares at time 0 pays the fee.
        charge_unwind: if True, the sale of all shares at expiry pays the fee.
            The shares are sold at expiry either way.

    Returns:
        A float: premium received, plus gains and losses on the shares, minus fees,
        minus the payoff owed at expiry.
    """
    n_steps = len(path) - 1
    t = n_days / TRADING_DAYS_PER_YEAR  # time to expiry at the start, in years
    dt = t / n_steps  # time between rebalances, in years

    cash = float(bs_call_price(path[0], strike, sigma, t))  # premium received at time 0
    shares = 0.0  # we start holding nothing

    # Trade at steps 0, 1, ..., n_steps - 1. The holding chosen at step i is kept until step i + 1.
    for i in range(n_steps):
        price = path[i]
        tau = (n_steps - i) * dt  # time left to expiry
        if hedge:
            target = float(bs_call_delta(price, strike, sigma, tau))
        else:
            target = 0.0
        traded = target - shares  # positive = buy, negative = sell
        fee = fee_rate * abs(traded) * price
        if i == 0 and not charge_initial:
            fee = 0.0
        cash = cash - traded * price - fee  # pay for the shares, then pay the fee
        shares = target

    # At expiry: sell all the shares at the final price, then pay the option's payoff.
    final_price = path[-1]
    fee = fee_rate * abs(shares) * final_price
    if not charge_unwind:
        fee = 0.0
    cash = cash + shares * final_price - fee
    payoff = max(final_price - strike, 0.0)
    return cash - payoff


def hedge_pnl(paths, strike, sigma, n_days, fee_rate,
              hedge=True, charge_initial=True, charge_unwind=True):
    """Vectorized version of hedge_pnl_single_path.

    Args:
        paths: array of shape (n_paths, n_steps + 1).
        Other arguments as in hedge_pnl_single_path.

    Returns:
        Array of shape (n_paths,), one final P&L per path.
    """
    n_paths = paths.shape[0]
    n_steps = paths.shape[1] - 1
    t = n_days / TRADING_DAYS_PER_YEAR
    dt = t / n_steps

    cash = bs_call_price(paths[:, 0], strike, sigma, t)  # premium, one per path
    shares = np.zeros(n_paths)

    # The loop is over time steps only. Every line inside works on all paths at once.
    for i in range(n_steps):
        price = paths[:, i]
        tau = (n_steps - i) * dt
        if hedge:
            target = bs_call_delta(price, strike, sigma, tau)
        else:
            target = np.zeros(n_paths)
        traded = target - shares
        fee = fee_rate * np.abs(traded) * price
        if i == 0 and not charge_initial:
            fee = np.zeros(n_paths)
        cash = cash - traded * price - fee
        shares = target

    final_price = paths[:, -1]
    fee = fee_rate * np.abs(shares) * final_price
    if not charge_unwind:
        fee = np.zeros(n_paths)
    cash = cash + shares * final_price - fee
    payoff = np.maximum(final_price - strike, 0.0)
    return cash - payoff

In [7]:
# Check: the loop and the vectorized versions agree on a few paths, to many decimal places.
# 50 paths, every frequency, with and without fees, and with both accounting flags on and off.
small = fine_paths[:50]
for n in FREQUENCIES:
    paths_n = coarsen_paths(small, n)
    for fee_rate in [0.0, FEE_RATE]:
        for flag in [True, False]:
            vec = hedge_pnl(paths_n, STRIKE, SIGMA, N_DAYS, fee_rate,
                            charge_initial=flag, charge_unwind=flag)
            loop = np.array([
                hedge_pnl_single_path(p, STRIKE, SIGMA, N_DAYS, fee_rate,
                                      charge_initial=flag, charge_unwind=flag)
                for p in paths_n
            ])
            assert np.allclose(vec, loop, rtol=0, atol=1e-10), (n, fee_rate, flag)

# Check: with no hedge, P&L equals the premium minus the payoff, exactly.
paths_21 = coarsen_paths(fine_paths, 21)
premium = float(bs_call_price(S0, STRIKE, SIGMA, N_DAYS / TRADING_DAYS_PER_YEAR))
payoff = np.maximum(paths_21[:, -1] - STRIKE, 0.0)
for fee_rate in [0.0, FEE_RATE]:
    pnl_no_hedge = hedge_pnl(paths_21, STRIKE, SIGMA, N_DAYS, fee_rate, hedge=False)
    assert np.allclose(pnl_no_hedge, premium - payoff, rtol=0, atol=1e-12)

# Check: raising the fee never raises the mean P&L.
fee_means = [hedge_pnl(paths_21, STRIKE, SIGMA, N_DAYS, f).mean() for f in [0.0, 0.001, 0.002]]
assert fee_means[0] >= fee_means[1] >= fee_means[2], fee_means

# Check: with no fees, the mean P&L is within 4 standard errors of zero (21 rebalances).
pnl_21 = hedge_pnl(paths_21, STRIKE, SIGMA, N_DAYS, 0.0)
std_err = pnl_21.std(ddof=1) / np.sqrt(len(pnl_21))
print(f"no-fee mean P&L {pnl_21.mean():.4f}, standard error {std_err:.4f}")
assert abs(pnl_21.mean()) < 4 * std_err

print("all checks passed")

no-fee mean P&L 0.0010, standard error 0.0042
all checks passed


## 4. Hand check

Work out one path with three rebalancing steps on paper. Write your numbers here, then assert that your code matches to the cent.

**Path:** S = 100, 102, 99, 101 on days 0, 7, 14 and 21. K = 100, σ = 0.20, T = 21/252 = 0.083333, fee rate 0.001.
Delta is N(d1) with d1 = [ln(S/K) + ½σ²τ] / (σ√τ).

| step | S | τ | σ√τ | ln(S/K) | d1 | delta |
|---|---|---|---|---|---|---|
| 0 | 100 | 0.083333 | 0.057735 | 0 | 0.028868 | 0.511515 |
| 1 | 102 | 0.055556 | 0.047140 | 0.019803 | 0.443647 | 0.671351 |
| 2 | 99 | 0.027778 | 0.033333 | −0.010050 | −0.284843 | 0.387882 |

**Premium:** d2 = −d1 = −0.028868, so C = 100·(N(0.028868) − N(−0.028868)) = 100·(0.511515 − 0.488485) = **2.302974**.

**Gains on the shares** (holding × price change over the interval):
- 0.511515 × (102 − 100) = 1.023030
- 0.671351 × (99 − 102) = −2.014054
- 0.387882 × (101 − 99) = 0.775764
- total = **−0.215260**

**Fees** (0.001 × |shares traded| × price):
- step 0: buy 0.511515 at 100 → 0.051151
- step 1: buy 0.159836 at 102 → 0.016303
- step 2: sell 0.283469 at 99 → 0.028063
- step 3 (unwind): sell 0.387882 at 101 → 0.039176
- total = **0.134693**

**Payoff owed:** max(101 − 100, 0) = **1.000000**

**P&L** = 2.302974 − 0.215260 − 0.134693 − 1.000000 = **0.953021**, so **0.95** to the cent.

This uses the "gains on shares" form. The code tracks cash instead, so the two are independent ways to the same number.

Normal CDF values come from scipy; the rest is by hand.

In [8]:
# One short path with three rebalancing steps: prices on days 0, 7, 14 and 21.
# Round numbers, so the working is easy to follow on paper.
hand_path = np.array([100.0, 102.0, 99.0, 101.0])
hand_fee = FEE_RATE
hand_steps = len(hand_path) - 1
hand_t = N_DAYS / TRADING_DAYS_PER_YEAR
hand_dt = hand_t / hand_steps

# Same steps as hedge_pnl_single_path, printed one by one.
hand_cash = float(bs_call_price(hand_path[0], STRIKE, SIGMA, hand_t))
hand_shares = 0.0
print(f"premium received at step 0: {hand_cash:.6f}")
print(f"{'step':>4} {'tau':>9} {'price':>8} {'delta':>9} {'traded':>10} {'fee':>9} {'cash':>11}")
for i in range(hand_steps):
    price = hand_path[i]
    tau = (hand_steps - i) * hand_dt
    delta = float(bs_call_delta(price, STRIKE, SIGMA, tau))
    traded = delta - hand_shares
    fee = hand_fee * abs(traded) * price
    hand_cash = hand_cash - traded * price - fee
    hand_shares = delta
    print(f"{i:>4} {tau:>9.6f} {price:>8.2f} {delta:>9.6f} {traded:>10.6f} {fee:>9.6f} {hand_cash:>11.6f}")

final_price = hand_path[-1]
unwind_fee = hand_fee * abs(hand_shares) * final_price
hand_cash = hand_cash + hand_shares * final_price - unwind_fee
hand_payoff = max(final_price - STRIKE, 0.0)
print(f"step 3: sell {hand_shares:.6f} shares at {final_price:.2f}, fee {unwind_fee:.6f}, "
      f"cash {hand_cash:.6f}")
print(f"payoff owed: {hand_payoff:.6f}")
print(f"final P&L: {hand_cash - hand_payoff:.6f}")

# The printout must agree with the function.
hand_pnl = hedge_pnl_single_path(hand_path, STRIKE, SIGMA, N_DAYS, hand_fee)
assert abs((hand_cash - hand_payoff) - hand_pnl) < 1e-12

# My paper working (markdown above) gives 0.953021. The code must match it to the cent.
my_paper_pnl = 0.953021
assert abs(hand_pnl - my_paper_pnl) < 0.005  # to the cent
print("hand check matches to the cent")

premium received at step 0: 2.302974
step       tau    price     delta     traded       fee        cash
   0  0.083333   100.00  0.511515   0.511515  0.051151  -48.899664
   1  0.055556   102.00  0.671351   0.159836  0.016303  -65.219276
   2  0.027778    99.00  0.387882  -0.283469  0.028063  -37.183891
step 3: sell 0.387882 shares at 101.00, fee 0.039176, cash 1.953020
payoff owed: 1.000000
final P&L: 0.953020
hand check matches to the cent


## 5. Hedging error against rebalancing frequency, with and without fees

In [9]:
def run(fine_paths, frequencies, fee_rates):
    """Mean and standard deviation of the hedged P&L for every frequency and fee rate.

    Every frequency is sliced from the same fine paths, so all rows use the same paths.

    Returns:
        A DataFrame with columns n, fee, mean, std (one row per frequency and fee rate).
    """
    rows = []
    for n in frequencies:
        paths_n = coarsen_paths(fine_paths, n)
        for fee_rate in fee_rates:
            pnl = hedge_pnl(paths_n, STRIKE, SIGMA, N_DAYS, fee_rate)
            rows.append({"n": n, "fee": fee_rate, "mean": pnl.mean(), "std": pnl.std(ddof=1)})
    return pd.DataFrame(rows)


def loglog_slope(df, min_n=0):
    """Slope of a straight-line fit of log(std) against log(n), on the no-fee rows with n >= min_n."""
    rows = df[(df["fee"] == 0.0) & (df["n"] >= min_n)]
    slope = np.polyfit(np.log(rows["n"]), np.log(rows["std"]), 1)[0]
    return slope


results = run(fine_paths, FREQUENCIES, [0.0, FEE_RATE])
print(results.to_string(index=False))
slope_all = loglog_slope(results)
slope_21 = loglog_slope(results, min_n=21)
print(f"log-log slope, all four points: {slope_all:.3f}")
print(f"log-log slope, n >= 21 only:    {slope_21:.3f}")

  n   fee      mean      std
  5 0.000  0.011028 0.844049
  5 0.001 -0.151390 0.856471
 21 0.000  0.001011 0.422493
 21 0.001 -0.242870 0.438727
 84 0.000 -0.002381 0.217742
 84 0.001 -0.394176 0.258229
336 0.000 -0.000735 0.109998
336 0.001 -0.683655 0.253830
log-log slope, all four points: -0.484
log-log slope, n >= 21 only:    -0.485


In [10]:
# Where the fees come from. Each fee part is the drop in P&L when that part is switched on,
# using the same paths and the charge_initial / charge_unwind flags.
fee_rows = []
for n in FREQUENCIES:
    paths_n = coarsen_paths(fine_paths, n)
    pnl_no_fee = hedge_pnl(paths_n, STRIKE, SIGMA, N_DAYS, 0.0)
    pnl_fee = hedge_pnl(paths_n, STRIKE, SIGMA, N_DAYS, FEE_RATE)
    pnl_no_first = hedge_pnl(paths_n, STRIKE, SIGMA, N_DAYS, FEE_RATE, charge_initial=False)
    pnl_no_unwind = hedge_pnl(paths_n, STRIKE, SIGMA, N_DAYS, FEE_RATE, charge_unwind=False)
    total_fee = pnl_no_fee - pnl_fee
    first_fee = (pnl_no_first - pnl_fee).mean()
    unwind_fee = (pnl_no_unwind - pnl_fee).mean()
    fee_rows.append({
        "n": n,
        "total fee mean": total_fee.mean(),
        "total fee std": total_fee.std(ddof=1),
        "first trade": first_fee,
        "unwind": unwind_fee,
        "in between": total_fee.mean() - first_fee - unwind_fee,
    })
print(pd.DataFrame(fee_rows).to_string(index=False))

  n  total fee mean  total fee std  first trade   unwind  in between
  5        0.162418       0.046587     0.051151 0.051363    0.059903
 21        0.243880       0.068877     0.051151 0.051273    0.141455
 84        0.391795       0.117969     0.051151 0.051010    0.289634
336        0.682920       0.222908     0.051151 0.051122    0.580647


## 6. Plots

Plot the standard deviation of P&L against rebalancing frequency, with and without fees. For the no-fee case, also plot it on log-log axes and report the slope. Save both plots as PNG files next to your notebook.

In [11]:
def plot_std(ax, df):
    """Std of P&L against n, with and without fees, on a log x axis."""
    for fee_rate, label in [(0.0, "no fee"), (FEE_RATE, f"fee rate {FEE_RATE}")]:
        rows = df[df["fee"] == fee_rate]
        ax.plot(rows["n"], rows["std"], marker="o", label=label)
    ax.set_xscale("log")
    ax.set_xticks(FREQUENCIES, labels=[str(n) for n in FREQUENCIES])
    ax.set_xlabel("rebalances over the month (n)")
    ax.set_ylabel("std of P&L")
    ax.set_title("Std of P&L vs rebalancing frequency")
    ax.legend()


def plot_loglog(ax, df, slope):
    """No-fee std of P&L against n on log-log axes, with the fitted slope in the title."""
    rows = df[df["fee"] == 0.0]
    ax.loglog(rows["n"], rows["std"], marker="o", label="no fee")
    ax.set_xticks(FREQUENCIES, labels=[str(n) for n in FREQUENCIES])
    ax.set_xlabel("rebalances over the month (n), log scale")
    ax.set_ylabel("std of P&L, log scale")
    ax.set_title(f"No fee, log-log: slope {slope:.3f}")
    ax.legend()


# The two PNGs the task README asks for. Saved only, not shown, to keep the notebook small.
fig, ax = plt.subplots(figsize=(6, 4.5))
plot_loglog(ax, results, slope_all)
fig.tight_layout()
fig.savefig(OUT_DIR / f"{GITHUB_USER}_no_fee.png", dpi=120)
plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 4.5))
plot_std(ax, results)
fig.tight_layout()
fig.savefig(OUT_DIR / f"{GITHUB_USER}_with_fee.png", dpi=120)
plt.close(fig)

## Write-up

### Accounting choices
- Sell one ATM call (K = 100) at its Black-Scholes price, received at time 0. Interest rate 0, fractional shares allowed.
- Trade at times 0, T/n, ..., (n-1)T/n at that time's price, to the Black-Scholes delta. Fee = rate x |shares traded| x price.
- `charge_initial = True` and `charge_unwind = True`: the first purchase and the final sale both pay the fee. Open question for `docs/engine.md`.
- All frequencies use the same 10,000 paths (one 1,680-step simulation, sliced). Everything comes from the config cell and `SEED = 0`.

### Results
10,000 paths, `SEED = 0`, premium 2.3030.

| n | mean, no fee | std, no fee | mean, fee 0.001 | std, fee 0.001 |
|---|---|---|---|---|
| 5 | 0.0110 | 0.8440 | −0.1514 | 0.8565 |
| 21 | 0.0010 | 0.4225 | −0.2429 | 0.4387 |
| 84 | −0.0024 | 0.2177 | −0.3942 | 0.2582 |
| 336 | −0.0007 | 0.1100 | −0.6837 | 0.2538 |

- No-fee log-log slope: **−0.484** using all four points, **−0.485** using n ≥ 21 only.
- Fee breakdown (the cell after the table):
  - The first trade and the unwind cost about 0.051 each at every n.
  - The fees in between are 0.060, 0.141, 0.290 and 0.581, so they double each time n quadruples from 21 on.
  - The spread of the total fee grows from 0.047 to 0.223.

### Predictions vs reality
- **A:** confirmed. Every no-fee mean is within about 1.3 standard errors of zero.
- **B:** close. Slope -0.484 (-0.485 for n >= 21). I have not tested whether the gap from -0.5 is noise.
- **C:** confirmed. Mean P&L falls from -0.15 to -0.68, the in-between fees grow like sqrt(n), and the std flattens (0.258 at n = 84, 0.254 at n = 336).

## 7. What surprised you

The no-fee spread halves almost exactly each time n quadruples, even starting at n = 5. With a 0.1% fee, going from 84 to 336 rebalances barely changes the spread (0.258 to 0.254) but costs about 0.29 more on average. Even rebalancing daily with no fees, the spread (0.42) is about 18% of the premium (2.30).